In [9]:
import lib_analise 
import lib_classificador

info_modelo_svm = lib_analise.get_info_modelo('svm')  # para garantir que a função está carregada da
dataset_svm = lib_analise.get_dataset_analise('svm')
lib_analise.print_informacao_analise('svm')


X_train = dataset_svm['X_train']
y_train = dataset_svm['y_train']
X_val = dataset_svm['X_val']
y_val = dataset_svm['y_val']
X_test = dataset_svm['X_test']
y_test = dataset_svm['y_test']

X_train_scaled = dataset_svm['X_train_scaled']
X_val_scaled = dataset_svm['X_val_scaled']
X_test_scaled = dataset_svm['X_test_scaled']


Nome do dataset:  svm
X_train.shape (9298, 48)
X_test.shape (6974, 48)
X_val.shape (6974, 48)
X_train_scaled.shape (9298, 48)
X_test_scaled.shape (6974, 48)
X_val_scaled.shape (6974, 48)
classes_mapping {'interf': np.int64(0), 'normal': np.int64(1)}
features_ganho_informacao ['mean_container_mem_pgpgin', 'mean_container_net_tx_packets', 'mean_os_mem_nr_mapped', 'mean_container_net_rx_packets', 'mean_os_mem_pgpgout', 'mean_os_disk_write_sectors', 'mean_os_disk_write_io', 'mean_container_cpu_user', 'mean_container_cpu_system', 'mean_container_mem_active_file', 'mean_os_mem_nr_active_file', 'mean_os_disk_time_in_queue', 'mean_container_mem_rss', 'mean_container_mem_inactive_anon', 'mean_container_mem_mapped_file', 'mean_os_disk_write_ticks', 'mean_os_mem_pgmajfault', 'mean_os_mem_pgpgin', 'mean_os_disk_io_ticks', 'mean_os_disk_read_ticks', 'mean_os_mem_nr_inactive_anon', 'mean_os_disk_write_merge', 'mean_container_disk_8:0_async', 'mean_container_mem_active_anon', 'mean_process_mem_shared

In [6]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np
import pandas as pd

# Extraindo os dados do dicionário

# Nota: DecisionTreeClassifier não requer normalização, então vamos usar os dados brutos
# Mas você pode trocar para X_train_scaled se preferir

# Inicializar o modelo
modelo = DecisionTreeClassifier(
    random_state=42,
    max_depth=10,  # Profundidade máxima da árvore
    min_samples_split=5,  # Mínimo de amostras para dividir um nó
    min_samples_leaf=2,  # Mínimo de amostras em um nó folha
    criterion='gini'  # Pode ser 'gini' ou 'entropy'
)

# Treinar o modelo
print("Treinando o modelo...")
modelo.fit(X_train, y_train)
print("Modelo: ",modelo)
print("Treinamento concluído!")

# Fazer predições
y_train_pred = modelo.predict(X_train)
y_val_pred = modelo.predict(X_val)
y_test_pred = modelo.predict(X_test)

# Avaliar o modelo
print("\n" + "="*50)
print("RESULTADOS DO TREINAMENTO")
print("="*50)

print(f"\nAcurácia no conjunto de TREINO: {accuracy_score(y_train, y_train_pred):.4f}")
print(f"Acurácia no conjunto de VALIDAÇÃO: {accuracy_score(y_val, y_val_pred):.4f}")
print(f"Acurácia no conjunto de TESTE: {accuracy_score(y_test, y_test_pred):.4f}")

# Relatório de classificação detalhado (conjunto de teste)
print("\n" + "="*50)
print("RELATÓRIO DE CLASSIFICAÇÃO (Conjunto de Teste)")
print("="*50)
print(classification_report(y_test, y_test_pred))

# Matriz de confusão
print("\n" + "="*50)
print("MATRIZ DE CONFUSÃO (Conjunto de Teste)")
print("="*50)
print(confusion_matrix(y_test, y_test_pred))

# Importância das features (se disponível)
if hasattr(X_train, 'columns'):
    print("\n" + "="*50)
    print("IMPORTÂNCIA DAS FEATURES")
    print("="*50)
    feature_importance = pd.DataFrame({
        'feature': X_train.columns,
        'importance': modelo.feature_importances_
    }).sort_values('importance', ascending=False)
    print(feature_importance)

Treinando o modelo...
Modelo:  DecisionTreeClassifier(max_depth=10, min_samples_leaf=2, min_samples_split=5,
                       random_state=42)
Treinamento concluído!

RESULTADOS DO TREINAMENTO

Acurácia no conjunto de TREINO: 0.9418
Acurácia no conjunto de VALIDAÇÃO: 0.9171
Acurácia no conjunto de TESTE: 0.9144

RELATÓRIO DE CLASSIFICAÇÃO (Conjunto de Teste)
              precision    recall  f1-score   support

           0       0.94      0.95      0.94      5089
           1       0.85      0.83      0.84      1885

    accuracy                           0.91      6974
   macro avg       0.89      0.89      0.89      6974
weighted avg       0.91      0.91      0.91      6974


MATRIZ DE CONFUSÃO (Conjunto de Teste)
[[4819  270]
 [ 327 1558]]

IMPORTÂNCIA DAS FEATURES
                             feature  importance
26              mean_os_disk_read_io    0.554564
21          mean_os_disk_write_merge    0.106009
10        mean_os_mem_nr_active_file    0.060053
34      mean_cont

In [10]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import xgboost as xgb
import pandas as pd

# Dicionário com os modelos disponíveis
available_classifiers = {
    'DecisionTree': DecisionTreeClassifier(criterion='entropy'),
    'KNN': KNeighborsClassifier(n_neighbors=3),
    'XGBoost': xgb.XGBClassifier(objective="binary:logistic", random_state=42),
}


# Dicionário para armazenar os resultados
resultados = {}
modelos_treinados = {}

# Loop para treinar cada modelo
for nome_modelo, modelo in available_classifiers.items():
    print("\n" + "="*70)
    print(f"TREINANDO: {nome_modelo}")
    print("="*70)
    
    # KNN funciona melhor com dados normalizados
    if nome_modelo == 'KNN':
        X_tr, X_vl, X_ts = X_train_scaled, X_val_scaled, X_test_scaled
    else:
        X_tr, X_vl, X_ts = X_train, X_val, X_test
    
    # Treinar o modelo
    print(f"Treinando {nome_modelo}...")
    modelo.fit(X_tr, y_train)
    print("✓ Treinamento concluído!")
    
    # Fazer predições
    y_train_pred = modelo.predict(X_tr)
    y_val_pred = modelo.predict(X_vl)
    y_test_pred = modelo.predict(X_ts)
    
    # Calcular acurácias
    acc_train = accuracy_score(y_train, y_train_pred)
    acc_val = accuracy_score(y_val, y_val_pred)
    acc_test = accuracy_score(y_test, y_test_pred)
    
    # Armazenar resultados
    resultados[nome_modelo] = {
        'acuracia_treino': acc_train,
        'acuracia_validacao': acc_val,
        'acuracia_teste': acc_test,
        'y_pred_test': y_test_pred
    }
    
    # Armazenar modelo treinado
    modelos_treinados[nome_modelo] = modelo
    
    # Exibir resultados
    print(f"\n📊 Resultados - {nome_modelo}:")
    print(f"   Acurácia TREINO:     {acc_train:.4f}")
    print(f"   Acurácia VALIDAÇÃO:  {acc_val:.4f}")
    print(f"   Acurácia TESTE:      {acc_test:.4f}")
    
    # Relatório de classificação
    print(f"\n📋 Relatório de Classificação ({nome_modelo}):")
    print(classification_report(y_test, y_test_pred))
    
    # Matriz de confusão
    print(f"\n🔢 Matriz de Confusão ({nome_modelo}):")
    print(confusion_matrix(y_test, y_test_pred))

# Resumo comparativo
print("\n" + "="*70)
print("📈 RESUMO COMPARATIVO DE TODOS OS MODELOS")
print("="*70)

df_resultados = pd.DataFrame({
    'Modelo': list(resultados.keys()),
    'Acurácia Treino': [resultados[m]['acuracia_treino'] for m in resultados.keys()],
    'Acurácia Validação': [resultados[m]['acuracia_validacao'] for m in resultados.keys()],
    'Acurácia Teste': [resultados[m]['acuracia_teste'] for m in resultados.keys()]
})

print(df_resultados.to_string(index=False))

# Identificar o melhor modelo
melhor_modelo_nome = max(resultados.items(), key=lambda x: x[1]['acuracia_teste'])[0]
melhor_acuracia = resultados[melhor_modelo_nome]['acuracia_teste']

print("\n" + "="*70)
print(f"🏆 MELHOR MODELO: {melhor_modelo_nome} (Acurácia Teste: {melhor_acuracia:.4f})")
print("="*70)


TREINANDO: DecisionTree
Treinando DecisionTree...
✓ Treinamento concluído!

📊 Resultados - DecisionTree:
   Acurácia TREINO:     1.0000
   Acurácia VALIDAÇÃO:  0.9269
   Acurácia TESTE:      0.9253

📋 Relatório de Classificação (DecisionTree):
              precision    recall  f1-score   support

           0       0.95      0.95      0.95      5089
           1       0.87      0.86      0.86      1885

    accuracy                           0.93      6974
   macro avg       0.91      0.90      0.90      6974
weighted avg       0.93      0.93      0.93      6974


🔢 Matriz de Confusão (DecisionTree):
[[4840  249]
 [ 272 1613]]

TREINANDO: KNN
Treinando KNN...
✓ Treinamento concluído!

📊 Resultados - KNN:
   Acurácia TREINO:     0.9667
   Acurácia VALIDAÇÃO:  0.9302
   Acurácia TESTE:      0.9316

📋 Relatório de Classificação (KNN):
              precision    recall  f1-score   support

           0       0.96      0.94      0.95      5089
           1       0.85      0.90      0.88  